In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from matplotlib.colors import LinearSegmentedColormap
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. Global Font and Style Configuration
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False 
plt.rcParams['mathtext.fontset'] = 'stix'

# ==========================================
# Size and Style Core Control Hub (Aligned with 14x12 system)
# ==========================================
CONFIG = {
    'fig_w': 14,             
    'fig_h': 12,             
    'dpi': 600,              

    'label_size': 32,        
    'tick_size': 22,         
    
    'cbar_label_size': 26,   
    'cbar_tick_size': 24,    

    'border_width': 2,       
    'tick_width': 2,         
    'tick_length': 8,        

    'xlabel_pad': 10,        
    'ylabel_pad': 15,        
    'cbar_pad': 15,          
}

# ==========================================
# 2. Parameter Settings
# ==========================================
S0, R0 = 1e7, 1e6
d, K_cap = 0.07, 1e9
Vsmax_range, Vrmax_range = (0.98, 1.0), (0.89, 0.90)
Vsmin, Vrmin = -0.1, -0.1
MICr_range, Kr_range = (64.0, 256.0), (1.0, 2.0)
gamma_ia_range = (0.95, 1.05)
gamma_ja_range = (0.8, 1.2)
U_range = (0.03, 0.12)

dt, total_time = 0.1, 24
time_steps = int(total_time / dt)

GRID_RES = 50
ks_range = np.linspace(4.0, 6.0, GRID_RES)
mics_range = np.linspace(2.0, 8.0, GRID_RES)
c_range = np.linspace(0, 20, 400)

# ==========================================
# 3. Core Simulation Function
# ==========================================
def run_heatmap_simulation_stratified(ks_vals, mics_vals, c_vals):
    # Recommended to adjust to 99 (33 low + 33 mid + 33 high) before publication to smooth noise
    repeats_per_point = 300 
    print(f"--- Generating clean heatmap (X=Ks, Y=MICs) ---")
    
    msc_grid = np.zeros((len(ks_vals), len(mics_vals)))
    c_reshaped = c_vals[:, None] 
    
    for i, ks_center in enumerate(ks_vals):
        if i % 10 == 0: print(f"Progress: Processing row {i}/{len(ks_vals)}...")
        for j, mics_center in enumerate(mics_vals):
            msc_repeats = []
            for k in range(repeats_per_point):
                # Dynamic stratified sampling logic (trisection based on repeats_per_point)
                third = repeats_per_point // 3
                if k < third: 
                    current_num_s, current_num_r = np.random.randint(5, 13), np.random.randint(1, 3)
                elif k < 2 * third: 
                    current_num_s, current_num_r = np.random.randint(13, 29), np.random.randint(2, 4)
                else: 
                    current_num_s, current_num_r = np.random.randint(29, 51), np.random.randint(4, 6)

                S = np.full((len(c_vals), current_num_s), S0, dtype=float)
                R = np.full((len(c_vals), current_num_r), R0, dtype=float)

                g_ia = np.random.uniform(*gamma_ia_range)
                g_ja = np.random.uniform(*gamma_ja_range)
                g_ib = g_ja + np.random.uniform(*U_range)
                
                curr_ks = np.random.uniform(ks_center*0.95, ks_center*1.05, current_num_s)
                curr_mics = np.random.uniform(mics_center*0.95, mics_center*1.05, current_num_s)
                
                term_r = (c_reshaped / np.random.uniform(*MICr_range, current_num_r)) ** np.random.uniform(*Kr_range, current_num_r)
                Vr = np.random.uniform(*Vrmax_range, current_num_r) - ((np.random.uniform(*Vrmax_range, current_num_r) - Vrmin) * term_r) / (term_r - (Vrmin / np.random.uniform(*Vrmax_range, current_num_r)))
                term_s = (c_reshaped / curr_mics) ** curr_ks
                Vs = np.random.uniform(*Vsmax_range, current_num_s) - ((np.random.uniform(*Vsmax_range, current_num_s) - Vsmin) * term_s) / (term_s - (Vsmin / np.random.uniform(*Vsmax_range, current_num_s))) 
                
                for _ts in range(time_steps):
                    pop = S.sum(axis=1, keepdims=True) + R.sum(axis=1, keepdims=True)
                    S += (Vs * S * (1.0 - g_ia * pop / K_cap) - d * S) * dt
                    R += (Vr * R * (1.0 - g_ib * pop / K_cap) - d * R) * dt
                    np.maximum(S, 0, out=S); np.maximum(R, 0, out=R)
                
                sc = (S.sum(axis=1)/S0/current_num_s) / (R.sum(axis=1)/R0/current_num_r + 1e-10)
                try: msc_repeats.append(np.interp(1.0, sc[::-1], c_vals[::-1]))
                except: pass
            msc_grid[i, j] = np.mean(msc_repeats) if msc_repeats else np.nan
    return msc_grid

msc_results = run_heatmap_simulation_stratified(ks_range, mics_range, c_range)

# ==========================================
# 4. Plotting (Clean borderless version)
# ==========================================
fig = plt.figure(figsize=(CONFIG['fig_w'], CONFIG['fig_h']), dpi=CONFIG['dpi'])
ax = fig.add_axes([0.12, 0.12, 0.72, 0.83])

# Original blue-red color gradient (Monet_Parliament)
original_colors = ["#2B3A67", "#637A9F", "#9A8F9F", "#C27D84", "#A64B4B"]
custom_cmap = LinearSegmentedColormap.from_list("Monet_Parliament", original_colors)

# Ensure the image fills the plot area without any padding
im = ax.imshow(msc_results.T, extent=[4.0, 6.0, 2.0, 8.0], 
               origin='lower', aspect='auto', cmap=custom_cmap, 
               interpolation='nearest')

ax.set_xlabel(r'$K_S$', fontsize=CONFIG['label_size'], labelpad=CONFIG['xlabel_pad'])
ax.set_ylabel(r'$MIC_S$', fontsize=CONFIG['label_size'], labelpad=CONFIG['ylabel_pad'])
ax.tick_params(axis='both', which='major', labelsize=CONFIG['tick_size'], 
               width=CONFIG['tick_width'], length=CONFIG['tick_length'])
for spine in ax.spines.values(): spine.set_linewidth(CONFIG['border_width'])

# Colorbar
cbar_ax = fig.add_axes([0.88, 0.12, 0.025, 0.83]) 
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=CONFIG['cbar_tick_size'], width=CONFIG['tick_width'])
cbar.set_label('MSC Value', fontsize=CONFIG['cbar_label_size'], labelpad=CONFIG['cbar_pad'])
cbar.outline.set_linewidth(CONFIG['border_width'])

# Save
save_dir = "./results_physiological"
if not os.path.exists(save_dir): os.makedirs(save_dir)
save_path = os.path.join(save_dir, "MSC_Heatmap_MIC_K_Clean.png")
plt.savefig(save_path, dpi=CONFIG['dpi'])
print(f"Clean heatmap successfully saved to: {save_path}")

plt.show()